In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Load datasets (adjust filenames if different)
df_rent = pd.read_csv("dataset for rent 1.csv")
df_sale = pd.read_csv("dataset for sale 1.csv")

print("Rent shape:", df_rent.shape)
print("Sale shape:", df_sale.shape)

Rent shape: (1000, 15)
Sale shape: (1000, 15)


In [2]:
df_rent = df_rent.drop(columns=['size_m2', 'listing_type'])
df_sale = df_sale.drop(columns=['size_m2', 'listing_type'])

df_rent = df_rent.rename(columns={'price': 'price_per_m2'})
df_sale = df_sale.rename(columns={'price': 'price_per_m2'})

print("Rent columns after cleaning:", df_rent.columns.tolist())

Rent columns after cleaning: ['latitude', 'longitude', 'location_name', 'price_per_m2', 'bedrooms', 'bathrooms', 'property_type', 'condition', 'near_school', 'near_hospital', 'near_market', 'parking', 'security_rating']


In [3]:
feature_columns = [
    'latitude', 'longitude', 'location_name', 'bedrooms', 'bathrooms',
    'property_type', 'condition',
    'near_school', 'near_hospital', 'near_market', 'parking', 'security_rating'
]

X_rent = df_rent[feature_columns]
y_rent = df_rent['price_per_m2']

X_sale = df_sale[feature_columns]
y_sale = df_sale['price_per_m2']

print("Rent features shape:", X_rent.shape)

Rent features shape: (1000, 12)


In [4]:
categorical_cols = ['location_name', 'property_type', 'condition']
numeric_cols = [
    'latitude', 'longitude', 'bedrooms', 'bathrooms',
    'near_school', 'near_hospital', 'near_market', 'parking', 'security_rating'
]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X_rent, y_rent, test_size=0.2, random_state=42)

rent_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

rent_pipeline.fit(X_train, y_train)

y_pred = rent_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- RENT MODEL ---")
print(f"MAE:  {mae:.2f} ETB/m²")
print(f"RMSE: {rmse:.2f} ETB/m²")
print(f"R²:   {r2:.4f}")

joblib.dump(rent_pipeline, 'rent_model_no_distance.pkl')
print("✅ Rent model saved as 'rent_model_no_distance.pkl'")


--- RENT MODEL ---
MAE:  19.57 ETB/m²
RMSE: 26.98 ETB/m²
R²:   0.9865
✅ Rent model saved as 'rent_model_no_distance.pkl'


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X_sale, y_sale, test_size=0.2, random_state=42)

sale_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

sale_pipeline.fit(X_train, y_train)

y_pred = sale_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- SALE MODEL ---")
print(f"MAE:  {mae:.2f} ETB/m²")
print(f"RMSE: {rmse:.2f} ETB/m²")
print(f"R²:   {r2:.4f}")

joblib.dump(sale_pipeline, 'sale_model_no_distance.pkl')
print("✅ Sale model saved as 'sale_model_no_distance.pkl'")


--- SALE MODEL ---
MAE:  351.71 ETB/m²
RMSE: 681.24 ETB/m²
R²:   0.9992
✅ Sale model saved as 'sale_model_no_distance.pkl'


In [7]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)


PREDICTION RESULT
Location: Kezira
Coordinates: (9.592311, 41.859803)
Bedrooms: 3, Bathrooms: 2
Property type: Villa, Condition: new
Area: 200 m²
------------------------------------------------------------
Rent per m²:     646.99 ETB
Total monthly rent: 129,398.00 ETB

Sale per m²:     75,660.00 ETB
Total sale price:  15,132,000.00 ETB


In [8]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 4,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)


PREDICTION RESULT
Location: Kezira
Coordinates: (9.592311, 41.859803)
Bedrooms: 4, Bathrooms: 2
Property type: Villa, Condition: new
Area: 200 m²
------------------------------------------------------------
Rent per m²:     735.12 ETB
Total monthly rent: 147,024.00 ETB

Sale per m²:     78,000.00 ETB
Total sale price:  15,600,000.00 ETB


In [9]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 5,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)


PREDICTION RESULT
Location: Kezira
Coordinates: (9.592311, 41.859803)
Bedrooms: 5, Bathrooms: 2
Property type: Villa, Condition: new
Area: 200 m²
------------------------------------------------------------
Rent per m²:     855.11 ETB
Total monthly rent: 171,022.00 ETB

Sale per m²:     80,074.46 ETB
Total sale price:  16,014,891.36 ETB


In [10]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)


PREDICTION RESULT
Location: Kezira
Coordinates: (9.592311, 41.859803)
Bedrooms: 3, Bathrooms: 2
Property type: Villa, Condition: new
Area: 200 m²
------------------------------------------------------------
Rent per m²:     646.99 ETB
Total monthly rent: 129,398.00 ETB

Sale per m²:     75,660.00 ETB
Total sale price:  15,132,000.00 ETB


In [ ]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 3,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)

In [ ]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 0,
    'near_hospital': 0,
    'near_market': 0,
    'parking': 0,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)

In [ ]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)

In [ ]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 0,
    'near_hospital': 0,
    'near_market': 0,
    'parking': 1,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)

In [ ]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 0,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)

In [ ]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 1,
    'near_market': 1,
    'parking': 1,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)

In [ ]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 4,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)


In [ ]:
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

test_property = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 0,
    'area_m2': 200
}

input_df = pd.DataFrame([{k: test_property[k] for k in feature_columns}])

rent_perm2 = rent_model.predict(input_df)[0]
sale_perm2 = sale_model.predict(input_df)[0]

total_rent = rent_perm2 * test_property['area_m2']
total_sale = sale_perm2 * test_property['area_m2']

print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Location: {test_property['location_name']}")
print(f"Coordinates: ({test_property['latitude']}, {test_property['longitude']})")
print(f"Bedrooms: {test_property['bedrooms']}, Bathrooms: {test_property['bathrooms']}")
print(f"Property type: {test_property['property_type']}, Condition: {test_property['condition']}")
print(f"Area: {test_property['area_m2']} m²")
print("-"*60)
print(f"Rent per m²:     {rent_perm2:,.2f} ETB")
print(f"Total monthly rent: {total_rent:,.2f} ETB")
print(f"\nSale per m²:     {sale_perm2:,.2f} ETB")
print(f"Total sale price:  {total_sale:,.2f} ETB")
print("="*60)

In [ ]:
import pandas as pd
import joblib

# Load the models (make sure the .pkl files are in the same folder as this notebook)
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

# Define the list of features in the exact order the model expects
feature_columns = [
    'latitude', 'longitude', 'location_name', 'bedrooms', 'bathrooms',
    'property_type', 'condition',
    'near_school', 'near_hospital', 'near_market', 'parking', 'security_rating'
]

# ------------------------------
# Example 1: Villa in Kezira, high security, all amenities
# ------------------------------
property1 = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 1,
    'near_market': 1,
    'parking': 1,
    'security_rating': 5,
    'area_m2': 200
}

# ------------------------------
# Example 2: Apartment in Sabian, lower security, no parking
# ------------------------------
property2 = {
    'latitude': 9.591185,
    'longitude': 41.85989,
    'location_name': 'Sabian',
    'bedrooms': 2,
    'bathrooms': 1,
    'property_type': 'Apartment',
    'condition': 'good',
    'near_school': 0,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 0,
    'security_rating': 3,
    'area_m2': 120
}

# ------------------------------
# Example 3: Land in Industrial Park (no bedrooms/bathrooms)
# ------------------------------
property3 = {
    'latitude': 9.614141,
    'longitude': 41.723603,
    'location_name': 'Dire Dawa Industrial Park',
    'bedrooms': 0,
    'bathrooms': 0,
    'property_type': 'Land',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 0,
    'parking': 0,
    'security_rating': 2,
    'area_m2': 500
}

# Function to predict and display results
def predict_and_display(property_data):
    # Prepare DataFrame
    input_df = pd.DataFrame([{k: property_data[k] for k in feature_columns}])
    
    # Predict
    rent_perm2 = rent_model.predict(input_df)[0]
    sale_perm2 = sale_model.predict(input_df)[0]
    
    # Calculate totals
    area = property_data['area_m2']
    total_rent = rent_perm2 * area
    total_sale = sale_perm2 * area
    
    # Display
    print("\n" + "="*60)
    print("🏠 PROPERTY DETAILS")
    print("="*60)
    for key, value in property_data.items():
        print(f"{key.replace('_', ' ').title()}: {value}")
    print("="*60)
    print("💰 PREDICTED PRICES")
    print("="*60)
    print(f"Rent per m²:      {rent_perm2:,.2f} ETB")
    print(f"Total monthly rent: {total_rent:,.2f} ETB")
    print(f"\nSale per m²:      {sale_perm2:,.2f} ETB")
    print(f"Total sale price:   {total_sale:,.2f} ETB")
    print("="*60)

# Run predictions for the three examples
predict_and_display(property1)
predict_and_display(property2)
predict_and_display(property3)

In [ ]:
import pandas as pd
import joblib

# Load the models (make sure the .pkl files are in the same folder as this notebook)
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

# Define the list of features in the exact order the model expects
feature_columns = [
    'latitude', 'longitude', 'location_name', 'bedrooms', 'bathrooms',
    'property_type', 'condition',
    'near_school', 'near_hospital', 'near_market', 'parking', 'security_rating'
]

# ------------------------------
# Example 1: Villa in Kezira, high security, all amenities
# ------------------------------
property1 = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 1,
    'near_market': 1,
    'parking': 1,
    'security_rating': 5,
    'area_m2': 200
}

# ------------------------------
# Example 2: Apartment in Sabian, lower security, no parking
# ------------------------------
property2 = {
    'latitude': 9.591185,
    'longitude': 41.85989,
    'location_name': 'Sabian',
    'bedrooms': 2,
    'bathrooms': 1,
    'property_type': 'Apartment',
    'condition': 'good',
    'near_school': 0,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 0,
    'security_rating': 3,
    'area_m2': 120
}

# ------------------------------
# Example 3: Land in Industrial Park (no bedrooms/bathrooms)
# ------------------------------
property3 = {
    'latitude': 9.614141,
    'longitude': 41.723603,
    'location_name': 'Dire Dawa Industrial Park',
    'bedrooms': 5,
    'bathrooms': 3,
    'property_type': 'House',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 2,
    'area_m2': 500
}

# Function to predict and display results
def predict_and_display(property_data):
    # Prepare DataFrame
    input_df = pd.DataFrame([{k: property_data[k] for k in feature_columns}])
    
    # Predict
    rent_perm2 = rent_model.predict(input_df)[0]
    sale_perm2 = sale_model.predict(input_df)[0]
    
    # Calculate totals
    area = property_data['area_m2']
    total_rent = rent_perm2 * area
    total_sale = sale_perm2 * area
    
    # Display
    print("\n" + "="*60)
    print("🏠 PROPERTY DETAILS")
    print("="*60)
    for key, value in property_data.items():
        print(f"{key.replace('_', ' ').title()}: {value}")
    print("="*60)
    print("💰 PREDICTED PRICES")
    print("="*60)
    print(f"Rent per m²:      {rent_perm2:,.2f} ETB")
    print(f"Total monthly rent: {total_rent:,.2f} ETB")
    print(f"\nSale per m²:      {sale_perm2:,.2f} ETB")
    print(f"Total sale price:   {total_sale:,.2f} ETB")
    print("="*60)

# Run predictions for the three examples
predict_and_display(property1)
predict_and_display(property2)
predict_and_display(property3)

In [ ]:
import pandas as pd
import joblib

# Load the models (make sure the .pkl files are in the same folder as this notebook)
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

# Define the list of features in the exact order the model expects
feature_columns = [
    'latitude', 'longitude', 'location_name', 'bedrooms', 'bathrooms',
    'property_type', 'condition',
    'near_school', 'near_hospital', 'near_market', 'parking', 'security_rating'
]

# ------------------------------
# Example 1: Villa in Kezira, high security, all amenities
# ------------------------------
property1 = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 1,
    'near_market': 1,
    'parking': 1,
    'security_rating': 5,
    'area_m2': 200
}

# ------------------------------
# Example 2: Apartment in Sabian, lower security, no parking
# ------------------------------
property2 = {
    'latitude': 9.591185,
    'longitude': 41.85989,
    'location_name': 'Sabian',
    'bedrooms': 2,
    'bathrooms': 1,
    'property_type': 'Apartment',
    'condition': 'good',
    'near_school': 0,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 0,
    'security_rating': 3,
    'area_m2': 120
}

# ------------------------------
# Example 3: Land in Industrial Park (no bedrooms/bathrooms)
# ------------------------------
property3 = {
    'latitude': 9.614141,
    'longitude': 41.723603,
    'location_name': 'Dire Dawa Industrial Park',
    'bedrooms': 5,
    'bathrooms': 3,
    'property_type': 'House',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 2,
    'area_m2': 120
}

# Function to predict and display results
def predict_and_display(property_data):
    # Prepare DataFrame
    input_df = pd.DataFrame([{k: property_data[k] for k in feature_columns}])
    
    # Predict
    rent_perm2 = rent_model.predict(input_df)[0]
    sale_perm2 = sale_model.predict(input_df)[0]
    
    # Calculate totals
    area = property_data['area_m2']
    total_rent = rent_perm2 * area
    total_sale = sale_perm2 * area
    
    # Display
    print("\n" + "="*60)
    print("🏠 PROPERTY DETAILS")
    print("="*60)
    for key, value in property_data.items():
        print(f"{key.replace('_', ' ').title()}: {value}")
    print("="*60)
    print("💰 PREDICTED PRICES")
    print("="*60)
    print(f"Rent per m²:      {rent_perm2:,.2f} ETB")
    print(f"Total monthly rent: {total_rent:,.2f} ETB")
    print(f"\nSale per m²:      {sale_perm2:,.2f} ETB")
    print(f"Total sale price:   {total_sale:,.2f} ETB")
    print("="*60)

# Run predictions for the three examples
predict_and_display(property1)
predict_and_display(property2)
predict_and_display(property3)

In [ ]:
import pandas as pd
import joblib

# Load the models (make sure the .pkl files are in the same folder as this notebook)
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

# Define the list of features in the exact order the model expects
feature_columns = [
    'latitude', 'longitude', 'location_name', 'bedrooms', 'bathrooms',
    'property_type', 'condition',
    'near_school', 'near_hospital', 'near_market', 'parking', 'security_rating'
]

# ------------------------------
# Example 1: Villa in Kezira, high security, all amenities
# ------------------------------
property1 = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 1,
    'near_market': 1,
    'parking': 1,
    'security_rating': 5,
    'area_m2': 200
}

# ------------------------------
# Example 2: Apartment in Sabian, lower security, no parking
# ------------------------------
property2 = {
    'latitude': 9.591185,
    'longitude': 41.85989,
    'location_name': 'Sabian',
    'bedrooms': 2,
    'bathrooms': 1,
    'property_type': 'Apartment',
    'condition': 'good',
    'near_school': 0,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 0,
    'security_rating': 3,
    'area_m2': 120
}

# ------------------------------
# Example 3: Land in Industrial Park (no bedrooms/bathrooms)
# ------------------------------
property3 = {
    'latitude': 9.614141,
    'longitude': 41.723603,
    'location_name': 'Dire Dawa Industrial Park',
    'bedrooms': 5,
    'bathrooms': 3,
    'property_type': 'House',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 2,
    'area_m2': 500
}

# Function to predict and display results
def predict_and_display(property_data):
    # Prepare DataFrame
    input_df = pd.DataFrame([{k: property_data[k] for k in feature_columns}])
    
    # Predict
    rent_perm2 = rent_model.predict(input_df)[0]
    sale_perm2 = sale_model.predict(input_df)[0]
    
    # Calculate totals
    area = property_data['area_m2']
    total_rent = rent_perm2 * area
    total_sale = sale_perm2 * area
    
    # Display
    print("\n" + "="*60)
    print("🏠 PROPERTY DETAILS")
    print("="*60)
    for key, value in property_data.items():
        print(f"{key.replace('_', ' ').title()}: {value}")
    print("="*60)
    print("💰 PREDICTED PRICES")
    print("="*60)
    print(f"Rent per m²:      {rent_perm2:,.2f} ETB")
    print(f"Total monthly rent: {total_rent:,.2f} ETB")
    print(f"\nSale per m²:      {sale_perm2:,.2f} ETB")
    print(f"Total sale price:   {total_sale:,.2f} ETB")
    print("="*60)

# Run predictions for the three examples
predict_and_display(property1)
predict_and_display(property2)
predict_and_display(property3)

In [ ]:
import pandas as pd
import joblib

# Load the models (make sure the .pkl files are in the same folder as this notebook)
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

# Define the list of features in the exact order the model expects
feature_columns = [
    'latitude', 'longitude', 'location_name', 'bedrooms', 'bathrooms',
    'property_type', 'condition',
    'near_school', 'near_hospital', 'near_market', 'parking', 'security_rating'
]

# ------------------------------
# Example 1: Villa in Kezira, high security, all amenities
# ------------------------------
property1 = {
    'latitude': 9.592311,
    'longitude': 41.859803,
    'location_name': 'Kezira',
    'bedrooms': 3,
    'bathrooms': 2,
    'property_type': 'Villa',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 1,
    'near_market': 1,
    'parking': 1,
    'security_rating': 5,
    'area_m2': 200
}

# ------------------------------
# Example 2: Apartment in Sabian, lower security, no parking
# ------------------------------
property2 = {
    'latitude': 9.591185,
    'longitude': 41.85989,
    'location_name': 'Sabian',
    'bedrooms': 2,
    'bathrooms': 1,
    'property_type': 'Apartment',
    'condition': 'good',
    'near_school': 0,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 0,
    'security_rating': 3,
    'area_m2': 120
}

# ------------------------------
# Example 3: Land in Industrial Park (no bedrooms/bathrooms)
# ------------------------------
property3 = {
    'latitude': 9.614141,
    'longitude': 41.723603,
    'location_name': 'Dire Dawa Industrial Park',
    'bedrooms': 3,
    'bathrooms': 1,
    'property_type': 'House',
    'condition': 'new',
    'near_school': 1,
    'near_hospital': 0,
    'near_market': 1,
    'parking': 1,
    'security_rating': 2,
    'area_m2': 80
}

# Function to predict and display results
def predict_and_display(property_data):
    # Prepare DataFrame
    input_df = pd.DataFrame([{k: property_data[k] for k in feature_columns}])
    
    # Predict
    rent_perm2 = rent_model.predict(input_df)[0]
    sale_perm2 = sale_model.predict(input_df)[0]
    
    # Calculate totals
    area = property_data['area_m2']
    total_rent = rent_perm2 * area
    total_sale = sale_perm2 * area
    
    # Display
    print("\n" + "="*60)
    print("🏠 PROPERTY DETAILS")
    print("="*60)
    for key, value in property_data.items():
        print(f"{key.replace('_', ' ').title()}: {value}")
    print("="*60)
    print("💰 PREDICTED PRICES")
    print("="*60)
    print(f"Rent per m²:      {rent_perm2:,.2f} ETB")
    print(f"Total monthly rent: {total_rent:,.2f} ETB")
    print(f"\nSale per m²:      {sale_perm2:,.2f} ETB")
    print(f"Total sale price:   {total_sale:,.2f} ETB")
    print("="*60)

# Run predictions for the three examples
predict_and_display(property1)
predict_and_display(property2)
predict_and_display(property3)

In [ ]:
# ============================================================
# MODEL PERFORMANCE GRAPHS
# Run this cell after all prediction cells above
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# reload models (safe to re-run)
rent_model = joblib.load('rent_model_no_distance.pkl')
sale_model = joblib.load('sale_model_no_distance.pkl')

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# collect CV predictions for both models
datasets = [
    ('RENT Model', rent_model, X_rent, y_rent, '#1976D2', '#90CAF9'),
    ('SALE Model', sale_model, X_sale, y_sale, '#E64A19', '#FFAB91'),
]

cv_results = {}
for label, model, X, y, c_dark, c_light in datasets:
    y_cv   = cross_val_predict(model, X, y, cv=kf)
    cv_r2  = cross_val_score(model, X, y, cv=kf, scoring='r2')
    cv_mae = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    model.fit(X, y)
    train_r2 = model.score(X, y)
    cv_results[label] = dict(
        y=y, y_cv=y_cv, cv_r2=cv_r2, cv_mae=cv_mae,
        train_r2=train_r2, c_dark=c_dark, c_light=c_light,
        model=model, X=X
    )

# ============================================================
# FIGURE 1 — Performance Dashboard (3 rows x 2 cols)
# ============================================================
fig = plt.figure(figsize=(18, 20))
fig.patch.set_facecolor('#F5F5F5')
fig.suptitle('Rent & Sale Price Prediction — Model Performance Dashboard',
             fontsize=17, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

for col, (label, res) in enumerate(cv_results.items()):
    y       = res['y']
    y_cv    = res['y_cv']
    c_dark  = res['c_dark']
    c_light = res['c_light']
    cv_r2   = res['cv_r2']
    train_r2 = res['train_r2']

    # Row 0: Actual vs Predicted
    ax = fig.add_subplot(gs[0, col])
    ax.set_facecolor('#FAFAFA')
    ax.scatter(y, y_cv, alpha=0.45, s=22, color=c_dark, edgecolors='none', label='Samples')
    mn, mx = float(y.min()), float(y.max())
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1.8, label='Perfect fit')
    r2_cv  = r2_score(y, y_cv)
    mae_cv = mean_absolute_error(y, y_cv)
    ax.set_title(f'{label}\nActual vs Predicted (5-Fold CV)', fontweight='bold', fontsize=12)
    ax.set_xlabel('Actual Price (ETB/m\u00b2)', fontsize=10)
    ax.set_ylabel('Predicted Price (ETB/m\u00b2)', fontsize=10)
    ax.text(0.04, 0.93, f'CV R\u00b2 = {r2_cv:.4f}\nCV MAE = {mae_cv:.2f} ETB',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=c_dark, alpha=0.9))
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.4)

    # Row 1: Residuals
    ax2 = fig.add_subplot(gs[1, col])
    ax2.set_facecolor('#FAFAFA')
    residuals = np.array(y) - np.array(y_cv)
    ax2.scatter(y_cv, residuals, alpha=0.45, s=22, color=c_dark, edgecolors='none')
    ax2.axhline(0, color='black', lw=1.8, linestyle='--')
    ax2.axhline(residuals.mean(), color='red', lw=1.2, linestyle=':',
                label=f'Mean residual = {residuals.mean():.2f}')
    ax2.set_title(f'{label}\nResiduals (Actual \u2212 Predicted)', fontweight='bold', fontsize=12)
    ax2.set_xlabel('Predicted Price (ETB/m\u00b2)', fontsize=10)
    ax2.set_ylabel('Residual', fontsize=10)
    ax2.legend(fontsize=9)
    ax2.grid(True, linestyle='--', alpha=0.4)

    # Row 2: R² per fold bar chart
    ax3 = fig.add_subplot(gs[2, col])
    ax3.set_facecolor('#FAFAFA')
    fold_labels = [f'Fold {i+1}' for i in range(5)]
    bars = ax3.bar(fold_labels, cv_r2, color=c_light, edgecolor=c_dark, linewidth=1.2)
    ax3.axhline(cv_r2.mean(), color=c_dark, lw=2, linestyle='--',
                label=f'Mean CV R\u00b2 = {cv_r2.mean():.4f}')
    ax3.axhline(train_r2, color='red', lw=1.8, linestyle=':',
                label=f'Train R\u00b2 = {train_r2:.4f}')
    for bar, val in zip(bars, cv_r2):
        ax3.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax3.set_ylim(max(0, cv_r2.min() - 0.06), 1.03)
    ax3.set_title(f'{label}\nR\u00b2 per Fold  (Train vs CV — overfit check)',
                  fontweight='bold', fontsize=12)
    ax3.set_ylabel('R\u00b2 Score', fontsize=10)
    ax3.legend(fontsize=9)
    ax3.grid(True, axis='y', linestyle='--', alpha=0.4)

plt.savefig('performance_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Figure 1 saved: performance_dashboard.png')

# ============================================================
# FIGURE 2 — Learning Curves
# ============================================================
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6))
fig2.patch.set_facecolor('#F5F5F5')
fig2.suptitle('Learning Curves — How Model Improves with More Data',
              fontsize=15, fontweight='bold')

for ax, (label, res) in zip(axes2, cv_results.items()):
    ax.set_facecolor('#FAFAFA')
    X, y    = res['X'], res['y']
    c_dark  = res['c_dark']
    c_light = res['c_light']
    model   = res['model']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    fracs = np.linspace(0.1, 1.0, 10)
    tr_scores, te_scores = [], []
    for frac in fracs:
        n = max(10, int(frac * len(X_tr)))
        model.fit(X_tr.iloc[:n], y_tr.iloc[:n])
        tr_scores.append(model.score(X_tr.iloc[:n], y_tr.iloc[:n]))
        te_scores.append(model.score(X_te, y_te))
    sizes = (fracs * len(X_tr)).astype(int)
    ax.plot(sizes, tr_scores, 'o-', color='#E53935', lw=2, label='Train R\u00b2')
    ax.plot(sizes, te_scores, 's-', color=c_dark,    lw=2, label='Test R\u00b2')
    ax.fill_between(sizes, tr_scores, te_scores, alpha=0.12, color='gray')
    ax.set_title(f'{label} — Learning Curve', fontweight='bold', fontsize=12)
    ax.set_xlabel('Training Samples Used', fontsize=10)
    ax.set_ylabel('R\u00b2 Score', fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.4)
    gap  = max(tr_scores) - te_scores[-1]
    note = f'Overfit gap = {gap:.3f}' if gap > 0.05 else f'Healthy gap = {gap:.3f}'
    ax.text(0.04, 0.08, note, transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=c_dark, alpha=0.9))

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Figure 2 saved: learning_curves.png')

# ============================================================
# FIGURE 3 — Feature Importance
# ============================================================
fig3, axes3 = plt.subplots(1, 2, figsize=(18, 7))
fig3.patch.set_facecolor('#F5F5F5')
fig3.suptitle('Feature Importance — What Drives the Price?',
              fontsize=15, fontweight='bold')

for ax, (label, res) in zip(axes3, cv_results.items()):
    ax.set_facecolor('#FAFAFA')
    model        = res['model']
    c_dark       = res['c_dark']
    c_light      = res['c_light']
    regressor    = model.named_steps['regressor']
    preprocessor = model.named_steps['preprocessor']
    cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(
        ['location_name', 'property_type', 'condition']
    )
    num_names = np.array(['latitude', 'longitude', 'bedrooms', 'bathrooms',
                          'near_school', 'near_hospital', 'near_market',
                          'parking', 'security_rating'])
    all_names   = np.concatenate([cat_names, num_names])
    importances = regressor.feature_importances_
    top_idx     = np.argsort(importances)[-15:]
    colors_bar  = [c_dark if i >= len(cat_names) else c_light for i in top_idx]
    ax.barh(all_names[top_idx], importances[top_idx],
            color=colors_bar, edgecolor='white', linewidth=0.6)
    ax.set_title(f'{label}\nTop 15 Feature Importances', fontweight='bold', fontsize=12)
    ax.set_xlabel('Importance Score', fontsize=10)
    ax.tick_params(axis='y', labelsize=9)
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor=c_dark,  label='Numeric feature'),
                       Patch(facecolor=c_light, label='Categorical feature')], fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Figure 3 saved: feature_importance.png')

# ============================================================
# FIGURE 4 — Metrics Comparison Bar Chart
# ============================================================
fig4, axes4 = plt.subplots(1, 3, figsize=(16, 5))
fig4.patch.set_facecolor('#F5F5F5')
fig4.suptitle('Model Metrics Comparison — Rent vs Sale',
              fontsize=14, fontweight='bold')

model_names = list(cv_results.keys())
bar_colors  = [cv_results[m]['c_dark'] for m in model_names]
r2_vals     = [cv_results[m]['cv_r2'].mean()  for m in model_names]
mae_vals    = [cv_results[m]['cv_mae'].mean() for m in model_names]
rmse_vals   = [np.sqrt(-cross_val_score(cv_results[m]['model'],
                        cv_results[m]['X'], cv_results[m]['y'],
                        cv=kf, scoring='neg_mean_squared_error')).mean()
               for m in model_names]

for ax, title, vals, fmt in zip(
    axes4,
    ['CV R\u00b2  (higher = better)',
     'CV MAE  ETB/m\u00b2  (lower = better)',
     'CV RMSE  ETB/m\u00b2  (lower = better)'],
    [r2_vals, mae_vals, rmse_vals],
    ['.4f', '.2f', '.2f']
):
    ax.set_facecolor('#FAFAFA')
    bars = ax.bar(model_names, vals, color=bar_colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.01,
                format(val, fmt), ha='center', va='bottom',
                fontsize=12, fontweight='bold')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_ylabel('Score', fontsize=10)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Figure 4 saved: metrics_comparison.png')

# ============================================================
# FINAL SUMMARY TABLE
# ============================================================
print('\n' + '='*58)
print('  FINAL MODEL EVALUATION SUMMARY')
print('='*58)
print(f"  {'Model':<20} {'CV R2':>8} {'CV MAE':>12} {'CV RMSE':>12}")
print('-'*58)
for name, r2, mae, rmse in zip(model_names, r2_vals, mae_vals, rmse_vals):
    print(f"  {name:<20} {r2:>8.4f} {mae:>12.2f} {rmse:>12.2f}")
print('='*58)
print('  Overfit gap < 0.05 = healthy  |  > 0.05 = overfitting')
print('='*58)
